In [ ]:
!pip install -U datasets huggingface_hub fsspec

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import math
import datasets
import requests

## Dataset and Tokenizer


In [ ]:
dataset = datasets.load_dataset("wmt/wmt14", "de-en", split=["train", "test"])

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

In [ ]:
df = pd.DataFrame(dataset[0]["translation"])

In [ ]:
unk_idx = 0
pad_idx = 1
sos_idx = 2
eos_idx = 3

### Train Tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace, Punctuation, Sequence as Pre_Seq
from tokenizers.normalizers import Lowercase, Sequence
from tokenizers.processors import TemplateProcessing
from tokenizers.normalizers import Sequence, NFC, Lowercase
from copy import deepcopy

tkz_en = Tokenizer(BPE(unk_token="[UNK]"))
tkz_en.normalizer = Sequence([NFC(), Lowercase()])
tkz_en.pre_tokenizer = Pre_Seq([Whitespace(), Punctuation()])
tkz_en.post_processor = TemplateProcessing(
    single="[SOS] $A [EOS]",
    special_tokens=[("[SOS]", sos_idx), ("[EOS]", eos_idx)],
)
tkz_de = deepcopy(tkz_en)
trainer = BpeTrainer(vocab_size=30000, special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"])

In [ ]:
training_data_de = df["de"].to_list()
training_data_en = df["en"].to_list()
import numpy as np
np.random.shuffle(training_data_de)
np.random.shuffle(training_data_en)

In [ ]:
tkz_en.train_from_iterator(training_data_en, trainer)
tkz_de.train_from_iterator(training_data_de, trainer)

In [ ]:
tkz_en.save("tokenizer_en.json")
tkz_de.save("tokenizer_de.json")

### Load Tokenizer

In [ ]:
text = requests.get("https://raw.githubusercontent.com/TheGEN1U5/DLventures/refs/heads/main/nlp/seq2seq/tokenizer_en.json").text
with open("tokenizer_en.json", "w") as f:
    f.write(text)
text = requests.get("https://raw.githubusercontent.com/TheGEN1U5/DLventures/refs/heads/main/nlp/seq2seq/tokenizer_de.json").text
with open("tokenizer_de.json", "w") as f:
    f.write(text)

In [ ]:
tkz_en = Tokenizer.from_file("tokenizer_en.json")
tkz_de = Tokenizer.from_file("tokenizer_de.json")

### Tokenize Data

In [ ]:
df["de_tokens"] = df["de"].apply(lambda x: torch.tensor(tkz_de.encode(x).ids, dtype=torch.long))
df["en_tokens"] = df["en"].apply(lambda x: torch.tensor(tkz_en.encode(x).ids, dtype=torch.long))
df["de_len"] = df["de"].apply(lambda x: len(x))
df["en_len"] = df["en"].apply(lambda x: len(x))

### Sampling Function

In [ ]:
from sklearn.model_selection import train_test_split
def sample(df, size, val_split):
    num_bins = 10  # increase/decrease depending on precision vs sample size tradeoff
    df['bin'] = pd.cut(df['en_len'], bins=num_bins)
    sampled_df = (
        df.groupby('bin', group_keys=False)
        .apply(lambda x: x.sample(frac=size / len(df), random_state=42))
    )
    train_df, val_df = train_test_split(
        sampled_df,
        test_size=val_split,  # 20% goes to val+test
        stratify=sampled_df['bin'],
        random_state=42
    )
    return train_df, val_df

### Create Loaders

In [ ]:
def collate_fn(batch):
    en, de = zip(*batch)
    padded_en = pad_sequence(en, batch_first=True, padding_value=pad_idx)
    padded_de = pad_sequence(de, batch_first=True, padding_value=pad_idx)
    return padded_en, padded_de

In [ ]:
def get_loader(df, batch_size):
    return DataLoader(list(zip(df["en_tokens"].to_numpy(), df["de_tokens"].to_numpy())), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

## Transformer


### Implementation

In [2]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len, embed_dim):
        super(PositionalEmbedding, self).__init__()
        self.embed_dim = embed_dim
        pos_mask = torch.zeros((max_seq_len, embed_dim))
        for pos in range(max_seq_len):
            for i in range(0, self.embed_dim, 2):
                pos_mask[pos, i] = np.sin(pos / 10000 ** (i / self.embed_dim))
                pos_mask[pos, i + 1] = np.cos(pos / 10000 ** (i / self.embed_dim))

        pos_mask = pos_mask.unsqueeze(0)
        self.register_buffer("pos_mask", pos_mask)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + torch.autograd.Variable(self.pos_mask[:,:seq_len], requires_grad=False)
        return x

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.e = embed_dim // num_heads
        self.d_k = key_dim
        self.d_v = value_dim
        self.k_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.q_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.v_proj = nn.ModuleList([nn.Linear(self.e, self.d_v) for _ in range(num_heads)])
        self.out = nn.Linear(self.num_heads * self.d_v, self.embed_dim)

    def forward(self, key, query, value, mask=None):
        self.mask = mask

        batch_size, seq_len = key.size(0), key.size(1)
        key = key.reshape(batch_size, seq_len, self.num_heads, self.e)
        query = query.reshape(batch_size, seq_len, self.num_heads, self.e)
        value = value.reshape(batch_size, seq_len, self.num_heads, self.e)

        k = torch.stack([proj(key[:, :, i, :]) for i, proj in enumerate(self.k_proj)], dim=2)  # [n, l, h, d_k]
        q = torch.stack([proj(query[:, :, i, :]) for i, proj in enumerate(self.q_proj)], dim=2)
        v = torch.stack([proj(value[:, :, i, :]) for i, proj in enumerate(self.v_proj)], dim=2)

        qkt_scaled = torch.einsum("nqhd,nkhd->nhqk", q, k) / math.sqrt(self.d_k)
        if mask is not None:
            qkt_scaled = qkt_scaled.masked_fill(mask == 0, float("-1e20"))
        sftmx = torch.softmax(qkt_scaled, dim=-1)

        scores = torch.einsum("nhqk,nkhd->nqhd", sftmx, v)
        concat = scores.reshape(scores.size(0), scores.size(1), -1)

        output = self.out(concat)
        return output

In [4]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Encoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)

    def forward(self, x):
        mha_output = self.MHA(x, x, x)
        add_norm1 = self.norm1(mha_output + self.drop1(x))
        ffn_output = self.ffn(add_norm1)
        output = self.norm2(ffn_output + self.drop2(add_norm1))
        return output

In [5]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Decoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.masked_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.cross_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm3 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)
        self.drop3 = nn.Dropout(0.1)

    def forward(self, x, enc_output):
        batch_size, trg_len, _ = x.shape
        # returns the lower triangular part of matrix filled with ones
        mask = torch.tril(torch.ones((trg_len, trg_len))).expand(
            batch_size, 1, trg_len, trg_len
        )
        masked_mha_output = self.masked_MHA(x, x, x, mask=mask)
        add_norm1 = self.norm1(masked_mha_output + self.drop1(x))
        cross_mha_output = self.cross_MHA(enc_output, add_norm1, enc_output)
        add_norm2 = self.norm2(cross_mha_output + self.drop2(add_norm1))
        ffn_output = self.ffn(add_norm2)
        output = self.norm3(ffn_output + self.drop3(add_norm2))
        return output

In [6]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, vocab_size_src, vocab_size_trg, key_dim, value_dim, num_heads, max_seq_len=100):
        super(Transformer, self).__init__()
        self.embedding_src = nn.Embedding(vocab_size_src, embed_dim, padding_idx=1)
        self.embedding_trg = nn.Embedding(vocab_size_src, embed_dim, padding_idx=1)
        self.pe = PositionalEmbedding(max_seq_len, embed_dim)
        self.encoder = Encoder(embed_dim, key_dim, value_dim, num_heads)
        self.decoder = Decoder(embed_dim, key_dim, value_dim, num_heads)
        self.out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, y):
        v_x = self.embedding_src(x)
        v_x = self.pe(v_x)
        v_y = self.embedding_trg(y)
        v_y = self.pe(v_y)
        enc_output = self.encoder(v_x)
        dec_output = self.decoder(v_y, enc_output)
        logits = self.out(dec_output)
        probs = torch.softmax(logits, dim=-1)

### Training loop

In [ ]:
def train(model, device, train_loader, val_loader, num_epochs=10, learning_rate=1e-3, pad_idx=0, early_stop=False):
    model.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

    train_losses = []
    val_losses = []

    best_val_loss = float('inf')
    patience_counter = 0
    patience_limit = 10
    best_model_state = None

    for epoch in range(num_epochs):
        # Training
        model.train()
        total_loss = 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            logits = model(src, tgt_input)

            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for src, tgt in val_loader:
                src = src.to(device)
                tgt = tgt.to(device)

                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]

                logits = model(src, tgt_input)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Early stopping logic
        if early_stop:
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    if best_model_state is not None:
                        model.load_state_dict(best_model_state)
                    return train_losses, val_losses

    if early_stop and best_model_state is not None:
        model.load_state_dict(best_model_state)

    return train_losses, val_losses


# Scaling Laws
https://arxiv.org/abs/2001.08361

## Law 1
### Performance with Non-Embedding Parameter Count N

$$
L(N) \approx \left( \frac{N_c}{N} \right) ^{α_N}
$$


According to this law, given that the dataset size and the compute budget is fixed, the Loss follows a power law of N (the number of non-embedding parameters)

**Calculation of Non-Embedding Parameters for basic transformers**
$$
N \approx 12n_{layer}d_{model}^2
$$

So, we will use 4 different transformer model sizes to demonstrate this power law.
The dataset size would be way less than the original paper and hence α may not completely match.

**Models Used**
-

In [9]:

model = Transformer(
    embed_dim=512,
    vocab_size_src=30000,
    vocab_size_tgt=30000,
    key_dim=256,
    value_dim=256,
    num_heads=8,
    max_seq_len=100
)

TypeError: Transformer.__init__() got an unexpected keyword argument 'vocab_size_tgt'

In [ ]:
# generate random embed_tensor
embed_tensor = torch.rand((500, 100))
vocab_size = 500
key_dim = 100
value_dim = 100
num_heads = 5
model = Transformer(100, vocab_size, embed_tensor, key_dim, value_dim, num_heads)

In [ ]:
x, y = torch.randint(0, vocab_size, (10, 100)), torch.randint(0, vocab_size, (10, 100))
print(x.shape, y.shape)

torch.Size([10, 100]) torch.Size([10, 100])


In [ ]:
model(x, y)

torch.Size([10, 100])